# Qwen3 — Tool Calling (MLX)

## Imports

In [1]:
import inspect
import json
import re
import typing
from pprint import pprint

import mlx_lm

print("mlx-lm:", mlx_lm.__version__)

mlx-lm: 0.31.3


## Load Model and Tokenizer

In [2]:
MODEL_ID = "mlx-community/Qwen3-1.7B-8bit"

model, tokenizer = mlx_lm.load(MODEL_ID)

print(f"Architecture: {model.model_type}")
print(f"Parameters: {mlx_lm.utils.get_total_parameters(model):,}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Architecture: qwen3
Parameters: 1,720,574,976


## Define Tools

In [3]:
def get_json_schema(func):
    docstring = inspect.getdoc(func) or ""
    main_desc = docstring.split("Args:")[0].strip()
    param_descs = {}
    args_match = re.search(r"Args:\s*\n((?:\s+.*\n?)+)", docstring)
    if args_match:
        args_block = args_match.group(1)
        param_matches = re.findall(
            r"^\s*([a-zA-Z_]\w*)\s*:\s*(.*?)(?=\n\s*[a-zA-Z_]\w*\s*:|\Z)",
            args_block,
            re.MULTILINE | re.DOTALL,
        )
        for name, desc in param_matches:
            param_descs[name] = re.sub(r"\s+", " ", desc.strip())

    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
    }

    type_hints = typing.get_type_hints(func)
    signature = inspect.signature(func)

    properties = {}
    required = []

    for name, param in signature.parameters.items():
        param_type = type_hints.get(name, str)
        json_type = type_map.get(param_type, "string")

        param_schema = {"type": json_type}
        if name in param_descs:
            param_schema["description"] = param_descs[name]
        properties[name] = param_schema

        if param.default == inspect.Parameter.empty:
            required.append(name)

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": main_desc,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }

In [4]:
def get_weather(location: str) -> str:
    """Get the current weather for a location.

    Args:
        location: The city and state, e.g. San Francisco, CA
    """
    return json.dumps({
        "location": location,
        "temperature": "68",
        "unit": "fahrenheit",
        "condition": "sunny",
    })


def get_stock_price(symbol: str) -> str:
    """Get the current stock price for a ticker symbol.

    Args:
        symbol: The stock ticker symbol, e.g. AAPL
    """
    return json.dumps({
        "symbol": symbol,
        "price": 294.38,
        "currency": "USD",
    })


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount of money from one currency to another.

    Args:
        amount: The amount of money to convert
        from_currency: The three-letter currency code to convert from, e.g. USD
        to_currency: The three-letter currency code to convert to, e.g. EUR
    """
    rate = 0.88
    return json.dumps({
        "converted_amount": round(amount * rate, 2),
        "to_currency": to_currency},
    )


AVAILABLE_TOOLS = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
}

tools = [get_json_schema(fn) for fn in AVAILABLE_TOOLS.values()]

pprint(tools, sort_dicts=False, width=120)

[{'type': 'function',
  'function': {'name': 'get_weather',
               'description': 'Get the current weather for a location.',
               'parameters': {'type': 'object',
                              'properties': {'location': {'type': 'string',
                                                          'description': 'The city and state, e.g. San Francisco, CA'}},
                              'required': ['location']}}},
 {'type': 'function',
  'function': {'name': 'get_stock_price',
               'description': 'Get the current stock price for a ticker symbol.',
               'parameters': {'type': 'object',
                              'properties': {'symbol': {'type': 'string',
                                                        'description': 'The stock ticker symbol, e.g. AAPL'}},
                              'required': ['symbol']}}},
 {'type': 'function',
  'function': {'name': 'convert_currency',
               'description': 'Convert an amount of money from

## Single Tool Call

In [5]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount of money from one currency to another.", "parameters": {"type": "object", "properties": {"amount": {"type": "number", "description": "The amount of money t

In [6]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

<tool_call>
{"name": "get_weather", "arguments": {"location": "San Francisco, CA"}}
</tool_call>


In [7]:
tool_calls = [
    json.loads(match)
    for match in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", response, re.DOTALL)
]
pprint(tool_calls, sort_dicts=False, width=120)

[{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}}]


In [8]:
call = tool_calls[0]
result = AVAILABLE_TOOLS[call["name"]](**call["arguments"])
print(result)

{"location": "San Francisco, CA", "temperature": "68", "unit": "fahrenheit", "condition": "sunny"}


In [9]:
messages.append({"role": "assistant", "content": "", "tool_calls": tool_calls})
messages.append({"role": "tool", "content": result})

pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the weather in San Francisco?"},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}}]},
 {'role': 'tool',
  'content': '{"location": "San Francisco, CA", "temperature": "68", "unit": "fahrenheit", "condition": "sunny"}'}]


In [10]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount of money from one currency to another.", "parameters": {"type": "object", "properties": {"amount": {"type": "number", "description": "The amount of money t

In [11]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

The weather in San Francisco is currently **68°F** and **sunny**.


## Multiple Tool Calls

In [12]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco and the stock price of AAPL?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

<tool_call>
{"name": "get_weather", "arguments": {"location": "San Francisco, CA"}}
</tool_call>
<tool_call>
{"name": "get_stock_price", "arguments": {"symbol": "AAPL"}}
</tool_call>


In [13]:
tool_calls = [
    json.loads(match)
    for match in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", response, re.DOTALL)
]
pprint(tool_calls, sort_dicts=False, width=120)

[{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}},
 {'name': 'get_stock_price', 'arguments': {'symbol': 'AAPL'}}]


In [14]:
messages.append({"role": "assistant", "content": "", "tool_calls": tool_calls})
for call in tool_calls:
    result = AVAILABLE_TOOLS[call["name"]](**call["arguments"])
    messages.append({"role": "tool", "content": result})

pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the weather in San Francisco and the stock price of AAPL?"},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}},
                 {'name': 'get_stock_price', 'arguments': {'symbol': 'AAPL'}}]},
 {'role': 'tool',
  'content': '{"location": "San Francisco, CA", "temperature": "68", "unit": "fahrenheit", "condition": "sunny"}'},
 {'role': 'tool', 'content': '{"symbol": "AAPL", "price": 294.38, "currency": "USD"}'}]


In [15]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount of money from one currency to another.", "parameters": {"type": "object", "properties": {"amount": {"type": "number", "description": "The amount of money t

In [16]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

The weather in San Francisco is 68°F, and the stock price of AAPL is $294.38.


## Sequential Tool Calls

In [17]:
messages = [
    {
        "role": "system",
        "content": (
            "You must call tools one at a time and wait for each result before deciding "
            "the next step. Never guess a tool argument that depends on a previous tool result."
        ),
    },
    {"role": "user", "content": "What is AAPL's stock price converted to EUR?"},
]

print(f"[user]\n{messages[-1]['content']}\n")

while True:
    chat = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)

    tool_calls = [
        json.loads(match)
        for match in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", response, re.DOTALL)
    ]
    if not tool_calls:
        break

    messages.append({"role": "assistant", "content": "", "tool_calls": tool_calls})
    for call in tool_calls:
        result = AVAILABLE_TOOLS[call["name"]](**call["arguments"])
        print(f"[tool call]\n{call['name']}({call['arguments']})\n")
        print(f"[tool response]\n{result}\n")
        messages.append({"role": "tool", "content": result})

print("[assistant]")
print(response)

[user]
What is AAPL's stock price converted to EUR?

[tool call]
get_stock_price({'symbol': 'AAPL'})

[tool response]
{"symbol": "AAPL", "price": 294.38, "currency": "USD"}

[tool call]
convert_currency({'amount': 294.38, 'from_currency': 'USD', 'to_currency': 'EUR'})

[tool response]
{"converted_amount": 259.05, "to_currency": "EUR"}

[assistant]
The stock price of AAPL in EUR is 259.05.


In [18]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'system',
  'content': 'You must call tools one at a time and wait for each result before deciding the next step. Never guess a '
             'tool argument that depends on a previous tool result.'},
 {'role': 'user', 'content': "What is AAPL's stock price converted to EUR?"},
 {'role': 'assistant', 'content': '', 'tool_calls': [{'name': 'get_stock_price', 'arguments': {'symbol': 'AAPL'}}]},
 {'role': 'tool', 'content': '{"symbol": "AAPL", "price": 294.38, "currency": "USD"}'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'name': 'convert_currency',
                  'arguments': {'amount': 294.38, 'from_currency': 'USD', 'to_currency': 'EUR'}}]},
 {'role': 'tool', 'content': '{"converted_amount": 259.05, "to_currency": "EUR"}'},
 {'role': 'assistant', 'content': 'The stock price of AAPL in EUR is 259.05.'}]


In [19]:
conversation = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=False,
)

print(conversation)

<|im_start|>system
You must call tools one at a time and wait for each result before deciding the next step. Never guess a tool argument that depends on a previous tool result.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amo